# Human Evaluation

Computes criterion means, overall score, and Krippendorff's alpha from raw ratings.


In [ ]:
%pip install -q pandas numpy krippendorff


In [ ]:
import pandas as pd
import numpy as np
import krippendorff
from pathlib import Path

P = Path("/content/human_evaluation.csv")
assert P.exists()

df = pd.read_csv(P)
criteria = ["correctness","clarity","sufficiency","helpfulness"]

evaluator_counts = df.groupby(["configuration","response_id"])["evaluator_id"].nunique()
assert (evaluator_counts == 3).all(), "Each response must contain ratings from three evaluators"

for criterion in criteria:
    assert df[criterion].between(1,5).all()

overall = (
    df.groupby("configuration")[criteria]
      .apply(lambda x: x.to_numpy(dtype=float).mean())
      .rename("overall_human_evaluation")
      .reset_index()
)

criterion_means = (
    df.groupby("configuration")[criteria]
      .mean()
      .reset_index()
)

display(criterion_means)
display(overall)


In [ ]:
alpha_rows = []
for config, g in df.groupby("configuration"):
    for criterion in criteria:
        pivot = g.pivot(
            index="evaluator_id",
            columns="response_id",
            values=criterion,
        )
        alpha = krippendorff.alpha(
            reliability_data=pivot.to_numpy(dtype=float),
            level_of_measurement="ordinal",
        )
        alpha_rows.append({
            "configuration":config,
            "criterion":criterion,
            "krippendorff_alpha":alpha,
        })

alpha_df = pd.DataFrame(alpha_rows)
display(alpha_df)
alpha_df.to_csv("/content/human_evaluation_alpha.csv", index=False)
